In [27]:
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
# import seaborn as sb
import numpy as np
import datetime
import netCDF4
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.ticker import ScalarFormatter
import cartopy.mpl.ticker as cticker
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde


%matplotlib qt

datasets_dir = Path('../../../data/datasets/hplc_world')
save_dir = Path('../../../reports/data')
pigments = ["chlide_a[mg*m^3]", "chla[mg*m^3]", "chlb[mg*m^3]", "chlc1+c2[mg*m^3]",
    "fucox[mg*m^3]", "19'hxfcx[mg*m^3]", "19'btfcx[mg*m^3]",
    "diadino[mg*m^3]", "allox[mg*m^3]", "diatox[mg*m^3]", "zeaxan[mg*m^3]",
    "beta_car[mg*m^3]", "peridinin[mg*m^3]"]
pigments_short = ['chlide', 'chla', 'chlb', 'chlc1 + c2', 'fucox', "19'hxfcx", "19'btfcx", 'diadino', 'allox', 'diatox', 'zeaxan', 'betac', 'peri']
pigments_short = ['chlide', 'chla', 'chlb', 'chlc12', 'fuco', "hex", "but", 'diad',
       'allo', 'diato', 'zea', 'caro', 'peri']
pigments_short_OLCI = ['chlide', 'chla', 'chlb', 'chlc1 + c2', 'fucox', "19'hxfcx", "19'btfcx", 'diadino', 'allox', 'diatox', 'zeaxan', 'peri']
pigments_short_OLCI = ['chlide', 'chla', 'chlb', 'chlc12', 'fuco', "hex", "but", 'diad',
       'allo', 'diato', 'zea', 'peri']
wv_11 = ['400', '412', '442', '490', '510', '560', '620', '665', '673', '681', '708']
wv_5 = ["412", "442", "490", "560", "673"]

save_dir.mkdir(parents=True, exist_ok=True)

In [28]:
x_OLCI = pd.read_csv(datasets_dir/'rrs_OLCI.csv')
x_OLCI[wv_11] = np.exp(x_OLCI[wv_11])

y_OLCI = pd.read_csv(datasets_dir/'hplc_OLCI.csv')
y_OLCI = y_OLCI.rename(columns=dict(zip(pigments, pigments_short)))
y_OLCI[pigments_short_OLCI] = np.exp(y_OLCI[pigments_short_OLCI])
print(len(y_OLCI))

x_multi = pd.read_csv(datasets_dir/'rrs_multi.csv')
x_multi[wv_5] = np.exp(x_multi[wv_5])

y_multi = pd.read_csv(datasets_dir/'hplc_multi.csv')
y_multi = y_multi.rename(columns=dict(zip(pigments, pigments_short)))
y_multi[pigments_short] = np.exp(y_multi[pigments_short])
print(len(y_multi))

668
7287


C:\Users\sheic\AppData\Local\Temp\ipykernel_9628\4191748931.py:12: DtypeWarning: Columns (1,79,133,134,226,227,228,229,230,231,232,233,234,235,236,237,282,283,353,354) have mixed types. Specify dtype option on import or set low_memory=False.
  y_multi = pd.read_csv(datasets_dir/'hplc_multi.csv')


In [29]:
x_OLCI.describe()

,Id,400,412,442,490,510,560,620,665,673,681,708,lat,lon
count,668.000000,668.000000,668.000000,668.000000,668.000000,668.000000,668.000000,668.000000,668.000000,668.000000,668.000000,668.000000,668.000000,668.000000
mean,39661.639222,0.004371,0.004760,0.006004,0.008419,0.008436,0.008666,0.003023,0.002037,0.001955,0.001937,0.001199,32.006673,-76.512534
std,21540.402781,0.004149,0.004351,0.005187,0.007395,0.007825,0.008984,0.004998,0.003966,0.003819,0.003800,0.002961,9.621875,37.185514
min,2788.000000,0.000164,0.000147,0.000163,0.000752,0.001074,0.000864,0.000035,0.000028,0.000018,0.000024,0.000005,9.104164,-167.479170
25%,8507.500000,0.001174,0.001514,0.002234,0.002962,0.002905,0.002193,0.000394,0.000196,0.000202,0.000200,0.000037,25.354166,-81.729164
50%,52635.500000,0.003248,0.003519,0.004502,0.005645,0.004714,0.004777,0.000954,0.000491,0.000491,0.000486,0.000233,26.729166,-81.187490
75%,52855.250000,0.006430,0.006841,0.008576,0.011817,0.011886,0.012360,0.003336,0.002020,0.001916,0.001886,0.001195,40.354168,-70.895836
max,53893.000000,0.026299,0.027840,0.033540,0.044320,0.046892,0.052700,0.034598,0.032347,0.032314,0.032337,0.031542,65.020836,110.937500


In [30]:
x_multi.describe()

,Id,412,442,490,560,673,lat,lon
count,7287.000000,7287.000000,7287.000000,7287.000000,7287.000000,7287.000000,7287.000000,7287.000000
mean,22087.149170,0.006539,0.005706,0.005287,0.003900,0.000950,23.816209,-99.501065
std,19167.034759,0.005242,0.003935,0.003444,0.004252,0.002037,27.849042,54.688585
min,171.000000,0.000197,0.000266,0.000768,0.000985,0.000026,-77.854164,-178.937500
25%,6397.500000,0.001785,0.002230,0.003029,0.001640,0.000140,21.229166,-157.187500
50%,15018.000000,0.004941,0.004998,0.005233,0.002290,0.000295,27.604166,-81.479164
75%,42273.500000,0.011600,0.009078,0.006424,0.004317,0.000865,40.145832,-70.875000
max,53893.000000,0.033866,0.036107,0.046535,0.052672,0.030828,75.979164,179.187520


In [31]:
y_OLCI.describe()

,Id,received,start_date,end_date,north_latitude,south_latitude,east_longitude,west_longitude,water_depth,missing,...,psp_20filt_bincount,tcar_20filt,tcar_20filt_bincount,tacc_20filt,tacc_20filt_bincount,tpg_20filt,tpg_20filt_bincount,dp_20filt,dp_20filt_bincount,number_of_data_rows
count,668.000000,6.680000e+02,6.680000e+02,6.680000e+02,668.000000,668.000000,668.000000,668.000000,542.000000,668.000000,...,19.0,19.000000,19.0,19.000000,19.0,19.000000,19.0,19.000000,19.0,64.000000
mean,39661.639222,2.021889e+07,2.018319e+07,2.018486e+07,33.538175,30.437852,-74.351086,-78.390661,-67.213911,-16951.095808,...,1.0,-2630.821684,1.0,-2630.582842,1.0,-2629.894895,1.0,-2630.832895,1.0,71.078125
std,21540.402781,2.070021e+04,2.139599e+04,1.867688e+04,9.569117,9.637397,36.767520,37.612103,419.138744,24432.033040,...,0.0,4523.990247,0.0,4524.136895,0.0,4524.559296,0.0,4523.983363,0.0,16.426174
min,2788.000000,2.017041e+07,2.014051e+07,2.016051e+07,13.000000,8.000000,-164.841183,-169.001690,-999.000000,-99999.000000,...,1.0,-9999.000000,1.0,-9999.000000,1.0,-9999.000000,1.0,-9999.000000,1.0,52.000000
25%,8507.500000,2.022073e+07,2.017033e+07,2.017033e+07,25.912000,24.478000,-80.380000,-83.100000,4.000000,-9999.000000,...,1.0,-4999.255000,1.0,-4999.143000,1.0,-4998.814500,1.0,-4999.266500,1.0,52.000000
50%,52635.500000,2.023082e+07,2.018081e+07,2.018081e+07,27.802000,24.478000,-80.380000,-82.210000,10.000000,-9999.000000,...,1.0,0.605000,1.0,0.912000,1.0,1.734000,1.0,0.608000,1.0,85.000000
75%,52855.250000,2.023082e+07,2.020011e+07,2.020011e+07,41.548000,37.697000,-69.958000,-74.951000,30.450000,-9999.000000,...,1.0,0.706000,1.0,1.042500,1.0,2.057500,1.0,0.694500,1.0,85.000000
max,53893.000000,2.023111e+07,2.023070e+07,2.023071e+07,65.000940,62.608067,111.000000,106.500000,2237.000000,-999.000000,...,1.0,0.872000,1.0,1.332000,1.0,2.617000,1.0,0.882000,1.0,85.000000


In [32]:
y_multi.describe()

,Id,start_date,end_date,north_latitude,south_latitude,east_longitude,west_longitude,water_depth,missing,lat,...,psp_20filt_bincount,tcar_20filt,tcar_20filt_bincount,tacc_20filt,tacc_20filt_bincount,tpg_20filt,tpg_20filt_bincount,dp_20filt,dp_20filt_bincount,number_of_data_rows
count,7287.000000,5.091000e+03,5.091000e+03,5091.000000,5091.000000,5091.000000,5091.000000,3168.000000,5.091000e+03,7287.000000,...,28.0,28.00000,28.0,28.000000,28.0,28.000000,28.0,28.000000,28.0,74.000000
mean,22087.149170,2.003840e+07,2.008112e+07,37.850883,26.980752,-68.634796,-111.527282,-622.742835,-7.778433e+32,23.814267,...,1.0,-2499.27125,1.0,-2499.038964,1.0,-2498.378429,1.0,-2499.283000,1.0,71.175676
std,19167.034759,9.718693e+04,5.380543e+04,12.303018,21.274565,26.952664,45.858212,512.034589,2.774186e+34,27.849332,...,0.0,4409.42604,0.0,4409.562613,0.0,4409.950982,0.0,4409.419132,0.0,16.392747
min,171.000000,1.992083e+07,1.997092e+07,-59.999900,-89.756200,-164.841183,-169.001690,-999.000000,-9.900000e+35,-77.865233,...,1.0,-9999.00000,1.0,-9999.000000,1.0,-9999.000000,1.0,-9999.000000,1.0,52.000000
25%,6397.500000,1.992083e+07,2.004121e+07,37.000000,18.649000,-75.700000,-166.378000,-999.000000,-9.999000e+03,21.220300,...,1.0,-2499.57600,1.0,-2499.486000,1.0,-2499.239250,1.0,-2499.575250,1.0,52.000000
50%,15018.000000,2.005041e+07,2.005042e+07,39.149000,25.999200,-69.916700,-85.402320,-999.000000,-9.990000e+02,27.583500,...,1.0,0.59200,1.0,0.872000,1.0,1.679000,1.0,0.570500,1.0,85.000000
75%,42273.500000,2.011032e+07,2.011071e+07,41.325000,39.782000,-56.653000,-74.033700,10.000000,-9.990000e+02,40.155900,...,1.0,0.68225,1.0,1.021750,1.0,1.944500,1.0,0.674750,1.0,85.000000
max,53893.000000,2.023070e+07,2.023071e+07,79.673800,70.703800,166.326800,106.500000,2237.000000,-9.990000e+02,75.965800,...,1.0,0.87200,1.0,1.332000,1.0,2.617000,1.0,0.882000,1.0,85.000000


# Data Description

In [35]:
fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.Mercator())  # You can choose different projections like 'PlateCarree'

# Add coastlines and other features
ax.coastlines()
ax.add_feature(cfeature.BORDERS, zorder=-10)
ax.add_feature(cfeature.LAND, zorder=-10, edgecolor='black')
# ax.add_feature(cfeature.LAKES, zorder=11, edgecolor='black')

# Set the extent for the Mediterranean Sea
# ax.set_extent([-180, 180, -80, 80], crs=ccrs.PlateCarree())  # [west, east, south, north]

ax.set_xlim(ax.projection.x_limits)
ax.set_ylim(ax.projection.y_limits)


# Plot the data as a scatter plot
sc = ax.scatter(x_multi['lon'], x_multi['lat'], c='firebrick', s=20, transform=ccrs.PlateCarree(), label=r'5-CENTER-WAVELENGTH')
sc = ax.scatter(x_OLCI['lon'], x_OLCI['lat'], c='royalblue', s=20, transform=ccrs.PlateCarree(), label=r'11-CENTER-WAVELENGTH')


# Add gridlines with labels
gl = ax.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=True,
    linewidth=0.8,
    color='gray',
    alpha=0.4,
    linestyle='--'
)

# Only label left and bottom axes (standard cartographic practice)
gl.top_labels = False
gl.right_labels = False

# Optional: control tick spacing
gl.xlocator = cticker.LongitudeLocator(10)
gl.ylocator = cticker.LatitudeLocator(5)


# Optional: format labels nicely
gl.xlabel_style = {'size': 22}
gl.ylabel_style = {'size': 22}

ax.text(
    0.5, -0.08, 'Longitude [degrees]',
    transform=ax.transAxes,
    ha='center', va='top',
    fontsize=25
)

ax.text(
    -0.1, 0.45, 'Latitude [degrees]',
    transform=ax.transAxes,
    ha='right', va='center',
    rotation=90,
    fontsize=25
)

plt.legend(fontsize=20, markerscale=4, frameon=True, facecolor="white", edgecolor="black", framealpha=0.9)
plt.tight_layout()
plt.savefig(save_dir / 'sample_location_world.jpg', bbox_inches='tight')
plt.show()


In [36]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.ticker import ScalarFormatter

# Use seaborn style for consistency with histograms
plt.style.use("seaborn-v0_8-whitegrid")

# Make sure wavelengths are numeric for plotting
wv_13 = ['400', '412', '442', '490', '510', '560', '620', '665', '673', '681', '708']
wv_13_int = [400, 412, 442, 490, 510, 560, 620, 665, 673, 681, 708]

fig, ax = plt.subplots(figsize=(8, 5))

# Convert to numpy for easier math
spectra = x_OLCI[wv_13].values
spectra_indx = np.max(spectra, axis=1)  < 0.04
spectra = spectra[spectra_indx]

# Compute mean and standard deviation spectra
mean_spectrum = np.mean(spectra, axis=0)
std_spectrum = np.std(spectra, axis=0)

#  separate by waters:
class1 = spectra[(np.argmax(spectra, axis=1)<=2)]
class2 = spectra[(np.argmax(spectra, axis=1)>2) * (np.argmax(spectra, axis=1)<=4)]
class3 = spectra[(np.argmax(spectra, axis=1)>4)]

# (Optional) Plot individual spectra as faint background lines
ax.plot(wv_13_int, class3.T,
    color="orange", alpha=0.4, linewidth=0.8, zorder=1
)


# (Optional) Plot individual spectra as faint background lines
ax.plot(wv_13_int, class2.T,
    color="green", alpha=0.4, linewidth=0.8, zorder=1
)

# (Optional) Plot individual spectra as faint background lines
ax.plot(wv_13_int, class1.T,
    color="blue", alpha=0.4, linewidth=0.8, zorder=1
)

# Plot mean spectrum (main curve)
# ax.plot(
#     wv_13, mean_spectrum,
#     color="steelblue", linewidth=2.5, label="Mean spectrum", zorder=2
# )

# Add ±1σ shading
# ax.fill_between(
#     wv_13,
#     mean_spectrum - std_spectrum,
#     mean_spectrum + std_spectrum,
#     color="steelblue", alpha=0.2, zorder=1.5,
#     label="±1σ"
# )

# Format Y-axis with scientific notation (10⁻² × ...)
# formatter = ScalarFormatter(useMathText=True)
# formatter.set_powerlimits((-2, -2))
# ax.yaxis.set_major_formatter(formatter)
# ax.ticklabel_format(axis='y', style='sci', scilimits=(-2, -2))
# ax.set_ylim(0, 0.015)
ax.set_xlim(400, 708)
# Axis labels and title
ax.set_xlabel("Wavelength [nm]", fontsize=22)
ax.set_ylabel(r"$R_{rs}$ $[sr^{-1}]$", fontsize=22)
# ax.set_title("Spectral Reflectance of All Samples", fontsize=14, weight="bold")

# Ticks and legend styling
ax.tick_params(labelsize=22)
# ax.legend(fontsize=10, loc="best", frameon=True, fancybox=True)
legend_lines = [
    Line2D([0], [0], color='blue', lw=5, label='Case-1'),
    Line2D([0], [0], color='green', lw=5, label='Case-2a'),
    Line2D([0], [0], color='orange', lw=5, label='Case-2b'),
]

leg = ax.legend(handles=legend_lines, fontsize=22, loc='best',    frameon=True, facecolor="white",edgecolor="black")# Final layout

frame = leg.get_frame()
frame.set_facecolor("white")
frame.set_edgecolor("black")
frame.set_alpha(1.0)
plt.tight_layout()
plt.savefig(save_dir / "spectral_plot_all_world_OLCI.jpg", dpi=300, bbox_inches="tight")
plt.show()



In [37]:
spectra.max(axis=0)

array([0.01910782, 0.02029697, 0.0244691 , 0.03464369, 0.03581809,
       0.03965677, 0.03171543, 0.03234749, 0.03231377, 0.03233718,
       0.03154175])

In [39]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.ticker import ScalarFormatter

# Use seaborn style for consistency with histograms
plt.style.use("seaborn-v0_8-whitegrid")

# Make sure wavelengths are numeric for plotting
wv_5_int = [412, 442, 490, 560, 673]

fig, ax = plt.subplots(figsize=(8, 5))

# Convert to numpy for easier math
spectra = x_multi[wv_5].values
spectra_indx = np.max(spectra, axis=1)  < 0.04
spectra = spectra[spectra_indx]

# Compute mean and standard deviation spectra
mean_spectrum = np.mean(spectra, axis=0)
std_spectrum = np.std(spectra, axis=0)

#  separate by waters:
class1 = spectra[(np.argmax(spectra, axis=1)<=0)]
class2 = spectra[(np.argmax(spectra, axis=1) == 2) + (np.argmax(spectra, axis=1) == 1)]
class3 = spectra[(np.argmax(spectra, axis=1)==3)]
# (Optional) Plot individual spectra as faint background lines
ax.plot(wv_5_int, class3.T,
    color="orange", alpha=0.4, linewidth=0.8, zorder=1
)
# (Optional) Plot individual spectra as faint background lines
ax.plot(wv_5_int, class2.T,
    color="green", alpha=0.4, linewidth=0.8, zorder=1
)
# (Optional) Plot individual spectra as faint background lines
ax.plot(wv_5_int, class1[-300:].T,
    color="blue", alpha=0.4, linewidth=0.8, zorder=1
)


# Plot mean spectrum (main curve)
# ax.plot(
#     wv_13, mean_spectrum,
#     color="steelblue", linewidth=2.5, label="Mean spectrum", zorder=2
# )

# Add ±1σ shading
# ax.fill_between(
#     wv_13,
#     mean_spectrum - std_spectrum,
#     mean_spectrum + std_spectrum,
#     color="steelblue", alpha=0.2, zorder=1.5,
#     label="±1σ"
# )

# Format Y-axis with scientific notation (10⁻² × ...)
# formatter = ScalarFormatter(useMathText=True)
# formatter.set_powerlimits((-2, -2))
# ax.yaxis.set_major_formatter(formatter)
# ax.ticklabel_format(axis='y', style='sci', scilimits=(-2, -2))
# ax.set_ylim(0, 0.015)
ax.set_xlim(412, 673)
# Axis labels and title
ax.set_xlabel("Wavelength [nm]", fontsize=22)
ax.set_ylabel(r"$R_{rs}$ $[sr^{-1}]$", fontsize=22)
# ax.set_title("Spectral Reflectance of All Samples", fontsize=14, weight="bold")

# Ticks and legend styling
ax.tick_params(labelsize=22)
# ax.legend(fontsize=10, loc="best", frameon=True, fancybox=True)
legend_lines = [
    Line2D([0], [0], color='blue', lw=5, label='Case-1'),
    Line2D([0], [0], color='green', lw=5, label='Case-2a'),
    Line2D([0], [0], color='orange', lw=5, label='Case-2b'),
]

leg = ax.legend(handles=legend_lines, fontsize=22, loc='best',    frameon=True, facecolor="white",edgecolor="black")# Final layout

frame = leg.get_frame()
frame.set_facecolor("white")
frame.set_edgecolor("black")
frame.set_alpha(1.0)
plt.tight_layout()
plt.savefig(save_dir / "spectral_plot_all_world_multi.jpg", dpi=300, bbox_inches="tight")
plt.show()


In [40]:
spectra.max(axis=0)

array([0.03386601, 0.02608177, 0.03518273, 0.0362641 , 0.0308279 ])

In [41]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, lognorm

bins = 30
# Optional: use a nice style
plt.style.use("seaborn-v0_8-whitegrid")

# Assuming your DataFrame is called `df` with 13 pigment columns
# e.g., df = pd.read_csv("pigments.csv")

df = y_OLCI[pigments_short_OLCI]
# Create a figure with subplots
fig, axes = plt.subplots(4, 3, figsize=(12, 16))  # 4x4 grid (1 empty)
axes = axes.flatten()

# Plot histograms
for i, col in enumerate(df.columns):
    pig_short = pigments_short_OLCI[i]
    ax = axes[i]
    df_filtered = df[df[col] <= 1]
    # --- Histogram ---
    sns.histplot(
        data=df_filtered,
        x=col,
        stat="percent",
        bins=bins,
        color="#4C72B0",
        edgecolor="white",
        linewidth=0.5,
        alpha=0.7,
        common_norm=False,
        ax=ax
    )

    # --- KDE scaled to percent ---
    data = df_filtered[col].dropna().values
    
    kde = gaussian_kde(data)
    xs = np.linspace(data.min(), data.max(), 500)
    
    bin_width = (data.max() - data.min()) / bins
    ys = kde(xs) * bin_width * 100
    
    ax.plot(xs, ys, color="firebrick", linewidth=2.5)
    # Calculating stats for the label
    # mu = df_filtered[col].mean()
    # sigma = df_filtered[col].std()
    
    stats_text = (f"$\\bf{{{pig_short}}}$")  # Bold Title
                  # f"$\\mu={mu:.2f}$\n"
                  # f"$\\sigma={sigma:.2f}$")
    
    ax.text(0.95, 0.95, stats_text, 
            transform=ax.transAxes, 
            fontsize=15, 
            verticalalignment='top', 
            horizontalalignment='right',
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9, edgecolor="#dddddd"))

    
    ax.set_xlabel("")  # remove x-label for cleanliness
    ax.set_ylabel("")  # remove x-label for cleanliness
    # ax.set_ylabel("KDE", fontsize=9)
    # ax.set_xlabel("Concentration [mg*m^3]", fontsize=9)
    ax.tick_params(labelsize=15)
    # ax.set_xlim(left=0)
    # 1. Fix the number of ticks on the left axis
    left_ticks = np.linspace(0, ax.get_ylim()[1], 5)   # 5 ticks: 0%, 25%, 50%, 75%, 100%
    left_ticks = np.round(left_ticks, 1)
    ax.set_yticks(left_ticks)
    
    # 2. Create the twin axis
    # ax2 = ax.twinx()
    
    # 3. Match the right axis ticks to the left axis count
    # patches = hist.patches
    # bin_width = patches[0].get_width()
    # right_ticks = left_ticks * bin_width * 100
    # ax2.set_ylim(0, right_ticks[-1])
    # right_ticks = np.round(right_ticks, 1)
    # ax2.set_yticks(right_ticks)

# Hide any empty subplots if 13 < 16
for j in range(len(df.columns), len(axes)):
    fig.delaxes(axes[j])

fig.text(0.5, 0.005, 'Pigment Concentration [mg $\cdot$ m$^{-3}$]', ha='center', fontsize=15)
fig.text(0.005, 0.5, 'Normalized Occurrences [%]', va='center', rotation='vertical', fontsize=15)
# fig.text(0.95, 0.5, 'Normalized Occurrences [%]', va='center', rotation=270, fontsize=12)

plt.tight_layout()
# Adjust rect to make room for global labels if needed
plt.subplots_adjust(bottom=0.06, left=0.06, right=0.91)

plt.savefig(save_dir / "pigment_histograms_world_OLCI.jpg", dpi=300, bbox_inches="tight")
plt.show()

In [42]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, lognorm

# Optional: use a nice style
plt.style.use("seaborn-v0_8-whitegrid")

# Assuming your DataFrame is called `df` with 13 pigment columns
# e.g., df = pd.read_csv("pigments.csv")

df = y_multi[pigments_short]
# Create a figure with subplots
fig, axes = plt.subplots(5, 3, figsize=(12, 16))  # 4x4 grid (1 empty)
axes = axes.flatten()

# Plot histograms
for i, col in enumerate(df.columns):
    pig_short = pigments_short[i]
    ax = axes[i]
    df_filtered = df[df[col] <= 100000000]
    hist = sns.histplot(
        data=df_filtered, 
        stat="percent",
        x=col, 
        kde=False,         # adds a smooth density curve
        bins=bins,          # adjust as needed
        color="#4C72B0",
        common_norm=False,
        edgecolor="white",
        linewidth=0.5,
        alpha=0.7,
        ax=ax
    )
    # ax.text(0.7, 0.9, pig_short, transform=ax.transAxes, fontsize=11, verticalalignment='top', bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=1))  # box styling)
    # sns.kdeplot(
    #     data=df_filtered, 
    #     x=col,
    #     color="firebrick",  # 👈 choose any dark red you like
    #     linewidth=1.5,
    #     ax=ax,
        # cut=0
    # )
    # --- KDE scaled to percent ---
    data = df_filtered[col].dropna().values
    
    kde = gaussian_kde(data)
    xs = np.linspace(data.min(), data.max(), 500)
    
    bin_width = (data.max() - data.min()) / bins
    ys = kde(xs) * bin_width * 100
    
    ax.plot(xs, ys, color="firebrick", linewidth=2.5)
    
    stats_text = (f"$\\bf{{{pig_short}}}$")  # Bold Title
                  # f"$\\mu={mu:.2f}$\n"
                  # f"$\\sigma={sigma:.2f}$")
    
    ax.text(0.95, 0.95, stats_text, 
            transform=ax.transAxes, 
            fontsize=15, 
            verticalalignment='top', 
            horizontalalignment='right',
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9, edgecolor="#dddddd"))

    
    ax.set_xlabel("")  # remove x-label for cleanliness
    ax.set_ylabel("")  # remove x-label for cleanliness
    # ax.set_ylabel("KDE", fontsize=9)
    # ax.set_xlabel("Concentration [mg*m^3]", fontsize=9)
    ax.tick_params(labelsize=15)
    # ax.set_xlim(left=0)
    # 1. Fix the number of ticks on the left axis
    left_ticks = np.linspace(0, ax.get_ylim()[1], 5)   # 5 ticks: 0%, 25%, 50%, 75%, 100%
    left_ticks = np.round(left_ticks, 1)
    ax.set_yticks(left_ticks)
    
    # 2. Create the twin axis
    # ax2 = ax.twinx()
    
    # 3. Match the right axis ticks to the left axis count
    # patches = hist.patches
    # bin_width = patches[0].get_width()
    # right_ticks = left_ticks * bin_width * 100
    # ax2.set_ylim(0, right_ticks[-1])
    # right_ticks = np.round(right_ticks, 1)
    # ax2.set_yticks(right_ticks)

# Hide any empty subplots if 13 < 16
for j in range(len(df.columns), len(axes)):
    fig.delaxes(axes[j])

fig.text(0.5, 0.005, 'Pigment Concentration [mg $\cdot$ m$^{-3}$]', ha='center', fontsize=15)
fig.text(0.005, 0.5, 'Normalized Occurrences [%]', va='center', rotation='vertical', fontsize=15)
# fig.text(0.95, 0.5, 'Normalized Occurrences [%]', va='center', rotation=270, fontsize=12)

plt.tight_layout()
# Adjust rect to make room for global labels if needed
plt.subplots_adjust(bottom=0.06, left=0.06, right=0.91)

plt.savefig(save_dir / "pigment_histograms_world_multi.jpg", dpi=300, bbox_inches="tight")
plt.show()

In [44]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Setup & Style
plt.style.use("seaborn-v0_8-whitegrid")

# Assuming x is your main dataframe and wv_13 is a list of column names
# df = x[wv_13] 
# FOR DEMO: I will create a dummy df. Replace this line with: df = x[wv_13]
import numpy as np
df = x_OLCI[wv_13]

# 2. Figure Setup
fig, axes = plt.subplots(4, 3, figsize=(12, 16)) 
axes = axes.flatten()

# 3. Plotting Loop
for i, col in enumerate(df.columns):
    ax = axes[i]
    
    # --- Data Filtering ---
    # Rrs values are usually small (< 0.1). 
    # If you have outliers/flags (like 999), filter them here.
    # Otherwise, just use df[col] directly.
    df_filtered = df[df[col] <= 1]  # Keeping your filter logic (adjust threshold if needed)

    # --- A. Histogram ---
    hist = sns.histplot(
        data=df_filtered, 
        stat="percent",
        x=col, 
        kde=False,
        bins=bins,          
        color="#4C72B0",    # Matches previous graph (Standard Seaborn blue)
        edgecolor="white",  # Matches previous graph
        linewidth=0.5,
        alpha=0.7,
        ax=ax
    )
    
    # --- B. KDE (Density Curve) ---
    # sns.kdeplot(
    #     data=df_filtered,
    #     x=col,
    #     color="firebrick",  # Matches previous graph
    #     linewidth=1.5,
    #     ax=ax,
    #     cut=0
    # )

    # --- C. Statistical Annotation Box ---
    # --- KDE scaled to percent ---
    data = df_filtered[col].dropna().values
    
    kde = gaussian_kde(data)
    xs = np.linspace(data.min(), data.max(), 500)
    
    bin_width = (data.max() - data.min()) / bins
    ys = kde(xs) * bin_width * 100
    
    ax.plot(xs, ys, color="firebrick", linewidth=2.5)
    # Using 'col' as the title. If you have a 'wavelengths_short' list, use that instead.
    stats_text = (f"$\\bf{{{col}}}nm$") 
                  # f"$\\mu={mu:.4f}$\n"  # Rrs is small, so used 4 decimal places
                  # f"$\\sigma={sigma:.4f}$")
    
    ax.text(0.95, 0.95, stats_text, 
            transform=ax.transAxes, 
            fontsize=15, 
            verticalalignment='top', 
            horizontalalignment='right',
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9, edgecolor="#dddddd"))

    # --- D. Formatting ---
    ax.set_xlabel("") 
    ax.set_ylabel("") 

    # Clean up ticks
    ax.tick_params(axis='both', which='major', labelsize=15)
    # ax.set_xlim(left=0) 
    left_ticks = np.linspace(0, ax.get_ylim()[1], 5)   # 5 ticks: 0%, 25%, 50%, 75%, 100%
    left_ticks = np.round(left_ticks, 1)
    ax.set_yticks(left_ticks)
    # 2. Create the twin axis
    # ax2 = ax.twinx()
    
    # 3. Match the right axis ticks to the left axis count
    # patches = hist.patches
    # bin_width = patches[0].get_width()
    # right_ticks = left_ticks * bin_width * 100
    # ax2.set_ylim(0, right_ticks[-1])
    # right_ticks = np.round(right_ticks, 1)
    # ax2.set_yticks(right_ticks)

# 4. Cleanup Empty Axes
for j in range(len(df.columns), len(axes)):
    fig.delaxes(axes[j])

# 5. Global Labels
# Note: Changed unit to steradian inverse (sr^-1) which is standard for Rrs. 
# Change back to mg m^-3 if this is actually concentration.
fig.text(0.5, 0.005, 'Remote Sensing Reflectance [sr$^{-1}$]', ha='center', fontsize=15)
fig.text(0.005, 0.5, 'Normalized Occurrences [%]', va='center', rotation='vertical', fontsize=15)
# fig.text(0.95, 0.5, 'Normalized Occurrences [%]', va='center', rotation=270, fontsize=12)

plt.tight_layout()
plt.subplots_adjust(bottom=0.06, left=0.06, right=0.91)

plt.savefig(save_dir / "rrs_histograms_world_OLCI.jpg", dpi=300, bbox_inches="tight")
plt.show()

In [45]:

# 1. Setup & Style
plt.style.use("seaborn-v0_8-whitegrid")

# Assuming x is your main dataframe and wv_13 is a list of column names
# df = x[wv_13] 
# FOR DEMO: I will create a dummy df. Replace this line with: df = x[wv_13]
import numpy as np
df = x_multi[wv_5]

# 2. Figure Setup
fig, axes = plt.subplots(2, 3, figsize=(15, 8)) 
axes = axes.flatten()

# 3. Plotting Loop
for i, col in enumerate(df.columns):
    ax = axes[i]
    
    # --- Data Filtering ---
    # Rrs values are usually small (< 0.1). 
    # If you have outliers/flags (like 999), filter them here.
    # Otherwise, just use df[col] directly.
    df_filtered = df[df[col] <= 1]  # Keeping your filter logic (adjust threshold if needed)

    # --- A. Histogram ---
    hist = sns.histplot(
        data=df_filtered, 
        stat="percent",
        x=col, 
        kde=False,
        bins=bins,          
        color="#4C72B0",    # Matches previous graph (Standard Seaborn blue)
        edgecolor="white",  # Matches previous graph
        linewidth=0.5,
        alpha=0.7,
        ax=ax
    )
    
    # --- B. KDE (Density Curve) ---
    # sns.kdeplot(
    #     data=df_filtered,
    #     x=col,
    #     color="firebrick",  # Matches previous graph
    #     linewidth=1.5,
    #     ax=ax,
    #     cut=0
    # )

    # --- C. Statistical Annotation Box ---
    # --- KDE scaled to percent ---
    data = df_filtered[col].dropna().values
    
    kde = gaussian_kde(data)
    xs = np.linspace(data.min(), data.max(), 500)
    
    bin_width = (data.max() - data.min()) / bins
    ys = kde(xs) * bin_width * 100
    
    ax.plot(xs, ys, color="firebrick", linewidth=2.5)
    # Using 'col' as the title. If you have a 'wavelengths_short' list, use that instead.
    stats_text = (f"$\\bf{{{col}}}nm$") 
                  # f"$\\mu={mu:.4f}$\n"  # Rrs is small, so used 4 decimal places
                  # f"$\\sigma={sigma:.4f}$")
    
    ax.text(0.95, 0.95, stats_text, 
            transform=ax.transAxes, 
            fontsize=15, 
            verticalalignment='top', 
            horizontalalignment='right',
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9, edgecolor="#dddddd"))

    # --- D. Formatting ---
    ax.set_xlabel("") 
    ax.set_ylabel("") 

    # Clean up ticks
    ax.tick_params(axis='both', which='major', labelsize=15)
    # ax.set_xlim(left=0) 
    left_ticks = np.linspace(0, ax.get_ylim()[1], 5)   # 5 ticks: 0%, 25%, 50%, 75%, 100%
    left_ticks = np.round(left_ticks, 1)
    ax.set_yticks(left_ticks)
    # 2. Create the twin axis
    # ax2 = ax.twinx()
    
    # 3. Match the right axis ticks to the left axis count
    # patches = hist.patches
    # bin_width = patches[0].get_width()
    # right_ticks = left_ticks * bin_width * 100
    # ax2.set_ylim(0, right_ticks[-1])
    # right_ticks = np.round(right_ticks, 1)
    # ax2.set_yticks(right_ticks)

# 4. Cleanup Empty Axes
for j in range(len(df.columns), len(axes)):
    fig.delaxes(axes[j])

# 5. Global Labels
# Note: Changed unit to steradian inverse (sr^-1) which is standard for Rrs. 
# Change back to mg m^-3 if this is actually concentration.
fig.text(0.5, 0.005, 'Remote Sensing Reflectance [sr$^{-1}$]', ha='center', fontsize=15)
fig.text(0.005, 0.5, 'Normalized Occurrences [%]', va='center', rotation='vertical', fontsize=15)
# fig.text(0.95, 0.5, 'Normalized Occurrences [%]', va='center', rotation=270, fontsize=12)

plt.tight_layout()
plt.subplots_adjust(bottom=0.06, left=0.06, right=0.91)

plt.savefig(save_dir / "rrs_histograms_world_multi.jpg", dpi=300, bbox_inches="tight")
plt.show()

# Data exploration

In [16]:
import pandas as pd

# Data extracted from Table S2
data = {
    "Pigment": [
        "19'-Butanoyloxyfucoxanthin", "19'-Hexanoyloxyfucoxanthin", "alpha-Carotene", "Alloxanthin",
        "beta-Carotene", "Chlorophyll b", "Chlorophyll c1", "Chlorophyll c2", "Chl c2-MGDG [14/18]",
        "Chlorophyll c3", "Chl c2-MGDG [14/14]", "Chlorophyllide a", "Cis-fucoxanthin",
        "Cis 19-hexanoyloxyfucoxanthin", "Diadinoxanthin", "Divinyl chlorophyll a", "Fucoxanthin",
        "Monovinyl chlorophyll a", "Monovinyl chlorophyll c3", "Monovinyl chlorophyll a allomer 1",
        "Monovinyl chlorophyll a allomer 2", "Monovinyl chlorophyll a epimer", "Neoxanthin",
        "Peridinin", "Prasinoxanthin", "Uriolide", "Violaxanthin", "Zeaxanthin"
    ],
    "Maximum (ng/l)": [
        136.37, 467.57, 49.96, 400.95, 58.45, 518.25, 41.81, 433.61, 109.75, 167.71,
        70.54, 609.08, 66.33, 43.45, 205.52, 47.95, 961.42, 3178.87, 26.91, 92.61,
        37.83, 22.75, 54.09, 222.72, 62.15, 40.4, 59.15, 104.27
    ],
    "Minimum (ng/l)": [
        1.25, 9.51, 0.1, 0.45, 1.46, 1.42, 0.05, 0.75, 0.26, 1.9, 0.53, 0.12, 0.0,
        0.43, 2.47, 0.1, 3.7, 58.03, 0.15, 0.08, 0.05, 0.49, 0.12, 0.47, 0.0, 0.0,
        0.22, 0.39
    ],
    "Average (ng/l)": [
        23.35, 73.36, 3.97, 18.79, 12.85, 57.05, 6.02, 55.66, 9.6, 31.77, 9.17,
        22.43, 5.88, 4.93, 30.42, 6.56, 90.31, 460.45, 3.88, 11.94, 7.83, 4.19,
        4.13, 8.24, 7.3, 3.53, 4.28, 21.24
    ],
    "SD (ng/l)": [
        18.21, 55.95, 4.75, 34.92, 8.23, 60.31, 7.13, 58.84, 16.7, 29.05, 9.74,
        67.11, 10.6, 5.88, 28.74, 9.24, 116.11, 375.5, 3.89, 12.12, 7.84, 3.8,
        5.23, 18.72, 8.53, 4.6, 5.42, 21.2
    ]
}

# Create a DataFrame
pigment_df = pd.DataFrame(data)

# Save to Excel
output_path = "HPLC_Pigment_Concentrations.xlsx"
pigment_df.to_csv(output_path, index=False)

print(f"Excel file saved successfully: {output_path}")


Excel file saved successfully: HPLC_Pigment_Concentrations.xlsx


In [12]:
pigment_hist = pd.read_csv('HPLC_Pigment_Concentrations.xlsx').set_index("Pigment")

In [13]:
pigment_hist

,Maximum (ng/l),Minimum (ng/l),Average (ng/l),SD (ng/l)
Pigment,,,,
19'-Butanoyloxyfucoxanthin,136.37,1.25,23.35,18.21
19'-Hexanoyloxyfucoxanthin,467.57,9.51,73.36,55.95
alpha-Carotene,49.96,0.10,3.97,4.75
Alloxanthin,400.95,0.45,18.79,34.92
beta-Carotene,58.45,1.46,12.85,8.23
Chlorophyll b,518.25,1.42,57.05,60.31
Chlorophyll c1,41.81,0.05,6.02,7.13
Chlorophyll c2,433.61,0.75,55.66,58.84
Chl c2-MGDG [14/18],109.75,0.26,9.60,16.70


In [14]:
from src.models.my_models import Model 

ImportError: cannot import name 'Model' from 'src.models.my_models' (C:\Users\sheic\PycharmProjects\pigment-retrieval-from-olci\src\models\my_models\__init__.py)

In [ ]:
x, y = pd.read_csv(Path('../../../data/datasets/dataset_hplc_multi/rrs_lat_lon_month_season_depth_loc.csv')), pd.read_csv(Path('../../../data/datasets/dataset_hplc_multi/log_pigments.csv'))

In [15]:
x.columns

Index(['400', '412', '442', '490', '510', '560', '620', '665', '673', '681',
       '708', '778', '865', 'lat', 'lon', 'January', 'February', 'March',
       'April', 'May', 'June', 'July', 'August', 'September', 'October',
       'November', 'December', 'spring', 'winter', 'autumn', 'summer', 'depth',
       'med', 'black sea', 'med and black sea'],
      dtype='object')

In [16]:
# Extract features of sample
y_variables = ['chlide_a[mg*m^3]', 'chla[mg*m^3]', 'chlb[mg*m^3]', 'chlc1+c2[mg*m^3]',
                    'fucox[mg*m^3]', "19'hxfcx[mg*m^3]", "19'btfcx[mg*m^3]", "diadino[mg*m^3]", "allox[mg*m^3]",
                    "diatox[mg*m^3]", "zeaxan[mg*m^3]", "beta_car[mg*m^3]", "peridinin[mg*m^3]"]

pigment_trad = {"19'btfcx[mg*m^3]": "19'-Butanoyloxyfucoxanthin", 
                "19'hxfcx[mg*m^3]": "19'-Hexanoyloxyfucoxanthin", 
                "allox[mg*m^3]": "Alloxanthin",
                "beta_car[mg*m^3]": "beta-Carotene", 
                'chlb[mg*m^3]': "Chlorophyll b", 
                'chlide_a[mg*m^3]': "Chlorophyllide a", 
                "diadino[mg*m^3]": "Diadinoxanthin", 
                'fucox[mg*m^3]': "Fucoxanthin", 
                "peridinin[mg*m^3]": "Peridinin", 
                "zeaxan[mg*m^3]": "Zeaxanthin"}
x_ds, y_ds = x[['400', '412', '442', '490', '510', '560', '620', '665', '673', '681',
       '708', '778', '865']],  y[y_variables]

In [25]:
y.describe()

,chlide_a[mg*m^3],chla[mg*m^3],chlb[mg*m^3],chlc1+c2[mg*m^3],fucox[mg*m^3],19'hxfcx[mg*m^3],19'btfcx[mg*m^3],diadino[mg*m^3],allox[mg*m^3],diatox[mg*m^3],zeaxan[mg*m^3],beta_car[mg*m^3],peridinin[mg*m^3]
count,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000
mean,0.162731,0.810745,0.045339,0.151623,0.318770,0.112247,0.016838,0.186608,0.028624,0.031112,0.038423,0.038602,0.030536
std,0.710862,1.673711,0.097423,0.469517,1.132058,0.113805,0.016789,0.688786,0.081034,0.069676,0.041310,0.141231,0.056404
min,0.000000,0.029000,0.001000,0.002000,0.001000,0.003000,0.002000,0.001000,0.000000,0.000300,0.003000,0.001000,0.000000
25%,0.005500,0.141200,0.005200,0.016400,0.009300,0.026700,0.005800,0.018800,0.002000,0.005000,0.015600,0.006300,0.003100
50%,0.011700,0.311200,0.022200,0.035200,0.032100,0.076600,0.011000,0.054800,0.006900,0.012700,0.028800,0.012900,0.006700
75%,0.023300,0.794300,0.043900,0.083400,0.121100,0.142600,0.019400,0.127300,0.024000,0.030400,0.043600,0.030000,0.028100
max,5.758500,16.615100,0.980600,4.571500,9.380900,0.519300,0.096800,7.969400,0.873200,0.673800,0.286900,1.795900,0.326500


In [22]:
x_ds.describe().to_markdown()

'|       |          400 |          412 |          442 |          490 |          510 |           560 |           620 |           665 |           673 |           681 |           708 |           778 |           865 |\n|:------|-------------:|-------------:|-------------:|-------------:|-------------:|--------------:|--------------:|--------------:|--------------:|--------------:|--------------:|--------------:|--------------:|\n| count | 185          | 185          | 185          | 185          | 185          | 185           | 185           | 185           | 185           | 185           | 185           | 185           | 185           |\n| mean  |   0.00501823 |   0.00558189 |   0.0061432  |   0.00659938 |   0.00560143 |   0.00433497  |   0.00131843  |   0.000885545 |   0.000865146 |   0.000860916 |   0.000584486 |   0.000218184 |   0.00016284  |\n| std   |   0.00197574 |   0.00233053 |   0.00250159 |   0.00272253 |   0.00283089 |   0.00358495  |   0.00180183  |   0.00121487  |   0.001146

In [ ]:
# corr = np.corrcoef(x_ds.values.T)
# fig = plt.figure(figsize = (5,5))

# sb.heatmap(corr, square = True)
# plt.show()
